# Final Analysis: Merge Results & Run Interventions

Run this notebook **after** all 3 sweep notebooks have completed:
- `07_sweep_cls.ipynb`
- `07_sweep_mean.ipynb`  
- `07_sweep_token.ipynb`

This notebook will:
1. Check sweep status
2. Re-evaluate checkpoints on validation set (if needed)
3. Merge all sweep results
4. Identify the best layer/pooling configuration
5. Run causal interventions on base model (with probe)
6. Run causal interventions on finetuned model
7. Generate comparison report and visualizations


## 1. Setup


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone or pull the repository
!git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
%cd Negation-Origin-Tracing


In [ ]:
# Install dependencies
%pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm


In [ ]:
# Set paths
import os
DRIVE_PATH = '/content/drive/MyDrive/NOT_results'
print(f"Looking for results in: {DRIVE_PATH}")


## 2. Check Sweep Status


In [ ]:
import json

sweeps = {
    'CLS': os.path.join(DRIVE_PATH, 'sweep_cls', 'results_cls.json'),
    'MEAN': os.path.join(DRIVE_PATH, 'sweep_mean', 'results_mean.json'),
    'TOKEN': os.path.join(DRIVE_PATH, 'sweep_token', 'results_token.json'),
}

print("Sweep Status:")
print("=" * 50)

all_complete = True
for name, path in sweeps.items():
    if os.path.exists(path):
        with open(path, 'r') as f:
            results = json.load(f)
        print(f"✓ {name}: {len(results)} experiments complete")
    else:
        print(f"✗ {name}: Not found")
        all_complete = False

if all_complete:
    print("\n✓ All sweeps complete! Ready to proceed.")
else:
    print("\n⚠ Some sweeps not complete. Wait for them to finish.")


## 3. Re-evaluate Checkpoints on Validation Set

**Note:** If the sweep results have `test_acc=0` and `test_auroc=0` (because SST-2 test set has -1 labels), 
run this cell to re-evaluate all checkpoints on the validation set and update the results files.


In [ ]:
# Re-evaluate checkpoints on validation set
# This updates test_acc/test_auroc with validation metrics
!python scripts/utils/reevaluate_sweep_results.py --drive_path {DRIVE_PATH} --data_dir data/raw


## 4. Merge Results & Find Best Configuration


In [ ]:
# Merge results and create visualizations
!python scripts/run_final_comparison.py --drive_path {DRIVE_PATH}


## 5. View Visualizations


In [ ]:
from IPython.display import Image, display

# Display heatmap
heatmap_path = os.path.join(DRIVE_PATH, 'final_comparison', 'probe_comparison.png')
if os.path.exists(heatmap_path):
    print("Probe Performance Heatmap:")
    display(Image(heatmap_path))

# Display bar chart  
bar_path = os.path.join(DRIVE_PATH, 'final_comparison', 'probe_comparison_bar.png')
if os.path.exists(bar_path):
    print("\nProbe Performance Bar Chart:")
    display(Image(bar_path))


## 6. Run Interventions: Base vs Finetuned Comparison


In [ ]:
# Download data if needed
if not os.path.exists('data/raw/train/sst.parquet'):
    !python src/data/download.py

# Run interventions comparison
!python scripts/run_interventions_comparison.py --drive_path {DRIVE_PATH}


## 7. View Comparison Report


In [ ]:
from IPython.display import Markdown

report_path = os.path.join(DRIVE_PATH, 'intervention_comparison', 'COMPARISON_REPORT.md')
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        display(Markdown(f.read()))
else:
    print("Report not found. Run interventions first.")
